In [23]:
#SETUP
from datetime import datetime
from delta import DeltaTable
import ConnectionConfig as cc
cc.setupEnvironment()

In [ ]:
#SETUP
spark = cc.startLocalCluster("dimCustomerIncrementalLoad")
spark.getActiveSession()

## Incremental load 
* Setting the parameters

In [24]:

run_timestamp = datetime.now()

In [25]:
cc.set_connectionProfile("VeloDB")

#### Read source table


In [26]:
#EXTRACT
df_velo_users =spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

df_subscriptions = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscriptions") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptionid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

df_subscription_types = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscription_types") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptiontypeid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

In [27]:
#TRANSFORM
df_velo_users.createOrReplaceTempView("operational_velo_users")
df_subscriptions.createOrReplaceTempView("operational_subscriptions")
df_subscription_types.createOrReplaceTempView("operational_subscription_types")

In [28]:
#TRANSFORM
df_dim_customers = spark.sql("""
    SELECT 
        uuid() as source_customer_SK,
        u.userid as source_customer_id,
        u.name as source_customer,
        CONCAT(u.street, ' ', u.number, ', ', u.zipcode, ' ', u.city, ', ', u.country_code) as source_address,
        st.description as source_subscription_type,
        s.validfrom as source_validform,
        md5(concat(u.name, source_address)) as source_md5
    FROM operational_velo_users u
        JOIN operational_subscriptions s 
                ON u.userid = s.userid
        JOIN operational_subscription_types st 
                ON s.subscriptiontypeid = st.subscriptiontypeid
    WHERE s.validfrom = (
        SELECT MAX(validfrom) 
        FROM operational_subscriptions 
        WHERE userid = u.userid)
""")

df_dim_customers.createOrReplaceTempView("dimCustomer_new")

In [29]:
#TRANSFORM
dt_dimCustomer = DeltaTable.forPath(spark, "./spark-warehouse/dimcustomer")
dt_dimCustomer.toDF().createOrReplaceTempView("dimCustomer_current")

#DEBUG CODE TO SHOW CONTENT OF DIMENSION
spark.sql("""
   select * from dimCustomer_new
""").show()

+--------------------+------------------+------------------+--------------------+------------------------+----------------+--------------------+
|  source_customer_SK|source_customer_id|   source_customer|      source_address|source_subscription_type|source_validform|          source_md5|
+--------------------+------------------+------------------+--------------------+------------------------+----------------+--------------------+
|4a8aff9c-fae5-4f7...|                 2| van der Zee Julia|Europalaan 43 , 2...|                   MAAND|      2021-10-05|b89ba72e92faa4269...|
|15a26c2a-6801-423...|                 1|       Bouman Lars|Somméstraat 156 ,...|                     DAG|      2020-12-14|16029dc8a664deffb...|
|205c445a-8fdc-490...|                 4|    Willems Angela|Graaf Joseph de P...|                     DAG|      2019-04-08|12722af9620035dbe...|
|5a953873-e8a7-455...|                 5|  Heijnen Patricia|Meylstraat 111 , ...|                    JAAR|      2022-05-16|e4ac667

##### 2 DETECT CHANGES


In [30]:
#TRANSFORM
detectedChanges = spark.sql("""
    select * 
    from dimCustomer_new source
        left outer join dimCustomer_current dwh 
            on dwh.userid = source.source_customer_id
            and dwh.current == true
    where dwh.userid is null or dwh.md5 <> source.source_md5
""")
detectedChanges.createOrReplaceTempView("detectedChanges")

#DEBUG CODE TO SHOW CONTENT OF DETECTED CHANGES
detectedChanges.show()

+--------------------+------------------+------------------+--------------------+------------------------+----------------+--------------------+-----------+------+------------------+--------------------+--------------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|  source_customer_SK|source_customer_id|   source_customer|      source_address|source_subscription_type|source_validform|          source_md5|     userSK|userId|              name|               email|             address| validfrom|description|          scd_start|            scd_end|                 md5|current|
+--------------------+------------------+------------------+--------------------+------------------------+----------------+--------------------+-----------+------+------------------+--------------------+--------------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|4a8aff9c-fae5-4f7...|                 2| van der

##### 3 TRANSOFRM TO UPSERTS


In [31]:
#TRANSFORM 
df_upsert = spark.sql("""
    select  
        source_customer_SK as customer_SK,
        source_customer_id as customer_id,
        source_customer as customer,
        source_address as address,
        source_subscription_type as subscription_type,
        source_validform as validfrom,
        to_timestamp('{run_timestamp}') as scd_start,
        to_timestamp('2600-01-01', 'yyyy-MM-dd') as scd_end,
        true as current,
        source_md5 as md5
    FROM detectedChanges
    union 
        select  
            userSK,
            userid,
            name,
            address,
            description,
            validfrom,
            scd_start,
            to_timestamp('{run_timestamp}') as scd_end,
            false,
            md5
        FROM detectedChanges
        where current is not null
""")

detectedChanges.createOrReplaceTempView("upsert")

In [32]:
#DEBUG CODE TO SHOW CONTENT OF UPSERT
spark.sql("""
    select * from upsert
""").show()

+--------------------+------------------+--------------------+--------------------+------------------------+----------------+--------------------+-----------+------+--------------------+--------------------+--------------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|  source_customer_SK|source_customer_id|     source_customer|      source_address|source_subscription_type|source_validform|          source_md5|     userSK|userId|                name|               email|             address| validfrom|description|          scd_start|            scd_end|                 md5|current|
+--------------------+------------------+--------------------+--------------------+------------------------+----------------+--------------------+-----------+------+--------------------+--------------------+--------------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|4a8aff9c-fae5-4f7...|               

In [35]:
#CREATE MERGE 
spark.sql("""
    MERGE INTO dimCustomer_current as target
    using upsert as source on target.userid = source_customer_id 
        and source.current = false 
        and target.current = true
    WHEN MATCHED THEN UPDATE SET scd_end = source.scd_end, current = source.current
    WHEN NOT MATCHED THEN INSERT (userSK, userid, name, address, description, validfrom, scd_start, scd_end, current, md5) values (source.userSK, source.userid, source.name, source.address, source.description, source.validfrom, source.scd_start, source.scd_end, source.current, source.md5)
""")

#DEBUG CODE TO SHOW CONTENT OF DIMENSION
dt_dimCustomer.toDF().sort("userid", "scd_start").show(100)

+-----------+------+--------------------+--------------------+--------------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|     userSK|userId|                name|               email|             address| validfrom|description|          scd_start|            scd_end|                 md5|current|
+-----------+------+--------------------+--------------------+--------------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|          2|     1|         Bouman Lars|Lars.Bouman@gmail...|Somméstraat 156  ...|2020-12-14|        DAG|1999-01-01 00:00:00|2100-12-12 00:00:00|8717594c222c27e77...|   true|
|          2|     1|         Bouman Lars|                NULL|Somméstraat 156  ...|2020-12-14|        DAG|1999-01-01 00:00:00|2100-12-12 00:00:00|8717594c222c27e77...|   true|
|          0|     2|   van der Zee Julia|Julia.van.der.Zee...|Europalaan 43  26...|2021-10-05|      MAAND|1999-01-01 00:

In [13]:
spark.stop()